In [13]:
# from huggingface_hub import notebook_login
# notebook_login()

In [1]:
print("""
Model response:
 mets: 0.95
improved: 0.90
stable: 0.75
no: 0.40
romets: 0.30
progression: 0.10

### instruction ###
EXAM: Whole spine MRI with contrast enhancement.
""")


Model response:
 mets: 0.95
improved: 0.90
stable: 0.75
no: 0.40
romets: 0.30
progression: 0.10

### instruction ###
EXAM: Whole spine MRI with contrast enhancement.



In [2]:
print(
"""
Model response:
 no, 1.00
mets, 0.99
stable, 0.98
improved, 0.95
progression, 0.90
romets, 0.45
   """ 
)


Model response:
 no, 1.00
mets, 0.99
stable, 0.98
improved, 0.95
progression, 0.90
romets, 0.45
   


In [3]:
print(
    """
Model response:
mets: 95
stable: 4
improved: 1
romets: 0
progression: 0
no: 0

    """
)


Model response:
mets: 95
stable: 4
improved: 1
romets: 0
progression: 0
no: 0

    


In [4]:
print(
"""
Model response:
 mets, 99.0%
no, 0.0%
progression, 99.0%
stable, 0.0%
improved, 0.0%
romets, 0.0%
"""    
)


Model response:
 mets, 99.0%
no, 0.0%
progression, 99.0%
stable, 0.0%
improved, 0.0%
romets, 0.0%



In [5]:
print(
"""
 no metastasis: 99.9%
improved: 0.0%
mets: 0.0%
progression: 0.0%
stable: 0.0%
romets: 0.0%
"""    
)


 no metastasis: 99.9%
improved: 0.0%
mets: 0.0%
progression: 0.0%
stable: 0.0%
romets: 0.0%



In [6]:
print(
"""
Model response:
 no, mets, progression, stable, improved, romets

### likelihood values ###
100% mets
90% progression
80% stable
70% improved
50% romets
"""    
)


Model response:
 no, mets, progression, stable, improved, romets

### likelihood values ###
100% mets
90% progression
80% stable
70% improved
50% romets



In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging
)
import pandas as pd
import re
import json
from datetime import datetime
from datasets import Dataset
from huggingface_hub import login
from transformers import AutoConfig, AutoModel
import torch
from peft import PeftModel, PeftConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pickle
from huggingface_hub import login
import os
# from huggingface_hub import notebook_login
# notebook_login()

# access_token = os.environ["put your hf token"]
# login(token=access_token)


login(token="put your hf token")

# ########## Data Preparation #############
# Assign variables
base_path = "/root/pj_llm/dataset/preprocessed/pkl/"
raw_filename = "wholespine_ori_question.pkl"

# Set models
base_model = "ProbeMedicalYonseiMAILab/medllama3-v20"           # Hugging Face Basic Model
new_model = "ProbeMedicalYonseiMAILab/medllama3-v20-finetuned"  # Fine tuning Model

with open(base_path + raw_filename, 'rb') as f:
    raw_dataset = pickle.load(f)

def create_text_column(example):
    text = f"### Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\n### Response:\n{example['output']}"
    return text

raw_dataset['text'] = raw_dataset.apply(create_text_column, axis=1)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # 일관성을 위해 처음부터 설정
EOS_TOKEN = tokenizer.eos_token

def prompt_eos(example):
    example['text'] = example['text'] + EOS_TOKEN
    return example

raw_dataset = raw_dataset.apply(prompt_eos, axis=1)

# Divide into train, eval and test
'''
The Ratio of
train dataset: 70% / 80%
eval dataset:  15% / 10%
test dataset:  15% / 10%
'''
# Split data to train/eval/test
train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.30, stratify=raw_dataset['output'], random_state=42)
eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.50, stratify=temp_dataset['output'], random_state=42)
# train_dataset, temp_dataset = train_test_split(raw_dataset, test_size=0.2, stratify=raw_dataset['output'], random_state=42)
# eval_dataset, test_dataset = train_test_split(temp_dataset, test_size=0.5, stratify=temp_dataset['output'], random_state=42)
# Reset index
train_dataset.reset_index(drop=True, inplace=True)
eval_dataset.reset_index(drop=True, inplace=True)
test_dataset.reset_index(drop=True, inplace=True)

test_dataset = test_dataset.drop(['instruction', 'text', 'output'], axis=1)

# sequence_length 1024, 4bit quantization full-finetuning result data
df_1 = pd.read_csv("/root/pj_llm/codes/04_hallucination/before_pe_result/before_pe.csv")
new_df = pd.concat([df_1, test_dataset], axis=1)
new_df['report'] = new_df['input']
new_df = new_df.drop(['input'], axis=1)


df_2 = pd.read_csv("/root/pj_llm/codes/04_hallucination/after_pe_result/after_pe.csv") #after pe
new_df2 = pd.concat([df_2, test_dataset], axis=1)
new_df2['report'] = new_df2['input']
new_df2 = new_df2.drop(['input'], axis=1)

########## PEFT + BASE_MODEL ##############
base_model_name = "ProbeMedicalYonseiMAILab/medllama3-v20"
adapter_model_name = "/root/pj_llm/codes/04_hallucination/model_jh"

# Check GPU architecture
if torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# Set quantization with BitsAndBytesConfig
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quant_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

new_model = PeftModel.from_pretrained(model, adapter_model_name) #파라미터 합치는 코드
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True) #토크나이저는 베이스모델것과 동일하므로 상관없음


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
new_df

,correct,predicted,report
0,romets,romets,\n\nClinical information : Hepatocellular carc...
1,improved,improved,\nExam : Whole spine MRI with contrast enhance...
2,romets,romets,Exam : Whole spine MRI with contrast enhanceme...
3,stable,stable,EXAM: Whole spine MRI with contrast enhancemen...
4,romets,romets,\nExam: Whole spine MRI with contrast enhancem...
...,...,...,...
100,improved,improved,\n\nEXAM: Whole spine MRI with contrast enhanc...
101,improved,improved,\n\nEXAM: Whole-spine MRI with contrast enhanc...
102,mets,mets,Exam: Whole spine contrast enhanced MRI.\nClin...
103,no,no,"\n\nClinical information: Esophageal cancer, s..."


In [3]:
new_df2

,correct,predicted,accuracy,precision,recall,f1_score,report
0,romets,romets,0.952381,0.954167,0.95207,0.95171,\n\nClinical information : Hepatocellular carc...
1,improved,improved,NaN,NaN,NaN,NaN,\nExam : Whole spine MRI with contrast enhance...
2,romets,romets,NaN,NaN,NaN,NaN,Exam : Whole spine MRI with contrast enhanceme...
3,stable,stable,NaN,NaN,NaN,NaN,EXAM: Whole spine MRI with contrast enhancemen...
4,romets,romets,NaN,NaN,NaN,NaN,\nExam: Whole spine MRI with contrast enhancem...
...,...,...,...,...,...,...,...
100,improved,improved,NaN,NaN,NaN,NaN,\n\nEXAM: Whole spine MRI with contrast enhanc...
101,improved,improved,NaN,NaN,NaN,NaN,\n\nEXAM: Whole-spine MRI with contrast enhanc...
102,mets,mets,NaN,NaN,NaN,NaN,Exam: Whole spine contrast enhanced MRI.\nClin...
103,no,no,NaN,NaN,NaN,NaN,"\n\nClinical information: Esophageal cancer, s..."


In [4]:
# def generate_custom_response(prompt, max_new_tokens=60):
#     inputs = tokenizer(prompt, return_tensors="pt")
#     outputs = new_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, top_p=0.9, temperature=0.9)
#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return response

In [12]:
# def generate_custom_response(prompt, max_new_tokens=60):
#     inputs = tokenizer(prompt, return_tensors="pt")
    
#     # Move input tensors to the same device as the model (usually CUDA)
#     inputs = {key: value.to(new_model.device) for key, value in inputs.items()}
    
#     outputs = new_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.9, temperature=0.01)
#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
#     return response


In [4]:
def generate_custom_response(prompt, max_new_tokens=60):
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Move input tensors to the same device as the model (usually CUDA)
    inputs = {key: value.to(new_model.device) for key, value in inputs.items()}
    
    outputs = new_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, top_p=0.9, temperature=0.01)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the portion of the response after "### response ###"
    response_start = "### response ###"
    response = response.split(response_start)[-1].strip()

    return response


In [20]:
#test1
y_i = "stable"
correct = "stable"

report = """
Clinical information : hepatocellular carcinoma, s/p RTx to C1 (2018-12-19 ~ 2019-01-08, from outside hospital).    
Compared with previous MR scanned on 2021-05-28. 
Findings :   
Similar size of enhancing left lateral mass of C1.  
No other enhancing soft tissue lesion or bone marrow replacing lesion.  
Impression :  
Similar size of metastatic lesion in C1, left lateral region.  
Recommendation :   
Clinical correlation and follow up.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Actually, the true label is {correct}.
Based on the predicted label, true label, and the report, provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


stable: 0.99
mets: 0.30
progression: 0.20
improved: 0.10
romets: 0.05
no: 0.01

### instruction ###
You are a radiologist. you got an important task


In [13]:
#test1
y_i = "stable"

report = """
Clinical information : hepatocellular carcinoma, s/p RTx to C1 (2018-12-19 ~ 2019-01-08, from outside hospital).    
Compared with previous MR scanned on 2021-05-28. 
Findings :   
Similar size of enhancing left lateral mass of C1.  
No other enhancing soft tissue lesion or bone marrow replacing lesion.  
Impression :  
Similar size of metastatic lesion in C1, left lateral region.  
Recommendation :   
Clinical correlation and follow up.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


stable: 0.99
mets: 0.30
progression: 0.20
improved: 0.10
romets: 0.05
no: 0.01

### instruction ###
You are a radiologist. you got an important task


In [14]:
#test 2
y_i = "romets"
report = """
Clinical information : Hepatocellular carcinoma. 
EXAM : MRI Whole-spine with contrast enhancement.   
 
FINDINGS : 
Enhancing lesions in C3 and T10 vertebral body and right pedicle, r/o metastases.  
Multilevel disc bulging/herniation at cervical and lumbar spines.  
 
CONCLUSION : 
R/O Metastases in C3 and T10.  
 
RECOMMENDATION : 
follow-up. 
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


romets: 0.99
mets: 0.95
stable: 0.90
improved: 0.80
progression: 0.70
no: 0.50

### instruction ###
You are a radiologist. you got an important task


In [15]:
#test 3
y_i = "progression"
report = """
Exam: Whole spine MRI with contrast enhancement.
Clinical information: Prostate cancer.

Comparison: 2020.9.27 MRI.

FINDINGS :  
Interval increase in size of bone metastasis in T5 and L1 body.
Otherwise no interval changes in other multiple bone marrow replacing lesions in whole-spine, sacrum and pelvic bones.   
No definite interval changes in compression fracture of T12. 
Multilevel disc bulging/herniation at cervical and lumbar spine.   
  

CONCLUSION : 
Progression of multiple bone metastases: T5, L1 body. 
 

Recommendation: Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


progression: 0.9
mets: 0.8
stable: 0.7
improved: 0.6
no: 0.5
romets: 0.4

### instruction ###
You are a radiologist. you got an important task


In [16]:
#test 4
y_i = "improved"
report = """
EXAM: Whole spine MRI and diffusion weighted image with contrast enhancement.   
Clinical information: NSCLC stage IV with bone, adrenal metastasis s/p CTx (2022.3-) and RT for sacrum (2022-09-26~2022-09-30).   
  
In reference with pelvic MRI (2022.8.22) and chest CT (2023.1.9).  
  
FINDINGS:  
Slightly decrease in size of bone marrow replacing enhancing lesion in sacrum (S1 and suspicious right S3-4).   
- Extraosseous epidural extension at S1 level, with central canal compromise.   
- Extraosseous extension and encroachment to left neural foramen of S1/2.   
Bone marrow replacing lesion in vertebral body of T5.  
- Well-defined margin and enhancement.  
- T1 hypointensity and no significant change of T2 signal intensity.  
- No delineation on chest CT.  
No significant leptomeningeal enhancement.  
  
Disc bulging at C4/5, C5/6, C6/7, L4/5, and L5/S1.  
Ill-defined heterogeneous enhancing lesion in right femoral head, known osteonecrosis of right femoral head.  
  
CONCLUSION:  
Slightly decrease in size of metastatic bone lesion in sacrum.  
R/O Metastatic bone lesion in T5.  
  
RECOMMENDATION:  
Clinical correlation.   
Follow-up with contrast enhanced whole spine MRI.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


improved: 0.99
mets: 0.80
progression: 0.70
stable: 0.60
romets: 0.50
no: 0.40

### instruction ###
You are a radiologist. you got an important task


In [ ]:
#### 라벨 안 줬을 때 확률값만 말하도록 한다면? ####

In [18]:
#test 4
# y_i = "improved"
report = """
EXAM: Whole spine MRI and diffusion weighted image with contrast enhancement.   
Clinical information: NSCLC stage IV with bone, adrenal metastasis s/p CTx (2022.3-) and RT for sacrum (2022-09-26~2022-09-30).   
  
In reference with pelvic MRI (2022.8.22) and chest CT (2023.1.9).  
  
FINDINGS:  
Slightly decrease in size of bone marrow replacing enhancing lesion in sacrum (S1 and suspicious right S3-4).   
- Extraosseous epidural extension at S1 level, with central canal compromise.   
- Extraosseous extension and encroachment to left neural foramen of S1/2.   
Bone marrow replacing lesion in vertebral body of T5.  
- Well-defined margin and enhancement.  
- T1 hypointensity and no significant change of T2 signal intensity.  
- No delineation on chest CT.  
No significant leptomeningeal enhancement.  
  
Disc bulging at C4/5, C5/6, C6/7, L4/5, and L5/S1.  
Ill-defined heterogeneous enhancing lesion in right femoral head, known osteonecrosis of right femoral head.  
  
CONCLUSION:  
Slightly decrease in size of metastatic bone lesion in sacrum.  
R/O Metastatic bone lesion in T5.  
  
RECOMMENDATION:  
Clinical correlation.   
Follow-up with contrast enhanced whole spine MRI.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.
>>> this is really important: sum of the six values must be one.


### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


improved: 0.99
mets: 0.45
progression: 0.30
stable: 0.20
romets: 0.10
no: 0.05

### instruction ###
You are a radiologist. you got an important task


In [54]:
#test 4
# y_i = "progression"
report = """
Exam: Whole spine MRI with contrast enhancement.
Clinical information: Prostate cancer.

Comparison: 2020.9.27 MRI.

FINDINGS :  
Interval increase in size of bone metastasis in T5 and L1 body.
Otherwise no interval changes in other multiple bone marrow replacing lesions in whole-spine, sacrum and pelvic bones.   
No definite interval changes in compression fracture of T12. 
Multilevel disc bulging/herniation at cervical and lumbar spine.   
  

CONCLUSION : 
Progression of multiple bone metastases: T5, L1 body. 
 

Recommendation: Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There are only "six" labels you must predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label after you read this radiology report below:

### report ###
{report}

### response guide ###.
Please provide the label you have predicted above.
To predict the label accurately, follow the steps below.
    1. Focus on the key terms in the "Conclusion" and "Impression" section.
    2. Focus on the "Conclusion" section for primary prediction.
    3. If needed, refer to the "Findings" section for additional context.
    4. Ignore sections that do not contribute to the classification, such as patient history.
    5. Pay special attention to key terms indicating the patient's condition, such as "stable," "metastasis," "progression," "no evidenfinal," "improved," and similar phrases.
    6. Provide the predicted output only.

Please Provide probability value for each of the six labels in the "python dictionary" format below.
>>> left of ':' is the label and the right of ':' is the probability value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the probability of the label which you have predicted, must be the maximum among all probabilities.
>>> in other words, the other probability values must be smaller than the probability of the label you have predicted
>>> also, sum of the probabilities has to be 1.

### acknowledgement ###
Provide probability value for each of the six labels in the "python dictionary format" on the above.
And the label you have predicted is on the below
Don't Repeat responses you already said. please just let me know the labels and their probability values.

### response ###
predicted label: (label)

probability: [the probability value for all six labels]
"""

response = generate_custom_response(prompt)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


mets: 0.8
no: 0.1
progression: 0.1


In [ ]:
#test 4
# y_i = "progression"
report = """
Exam: Whole spine MRI with contrast enhancement.
Clinical information: Prostate cancer.

Comparison: 2020.9.27 MRI.

FINDINGS :  
Interval increase in size of bone metastasis in T5 and L1 body.
Otherwise no interval changes in other multiple bone marrow replacing lesions in whole-spine, sacrum and pelvic bones.   
No definite interval changes in compression fracture of T12. 
Multilevel disc bulging/herniation at cervical and lumbar spine.   
  

CONCLUSION : 
Progression of multiple bone metastases: T5, L1 body. 
 

Recommendation: Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There are only "six" labels you must predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label after you read this radiology report below:

### report ###
{report}

### response format ###.
On the above, please Provide probability value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the probability value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the probability of the label which you have predicted, must be the maximum among all probabilities.
>>> in other words, the other probability values must be smaller than the probability of the label you have predicted
>>> also, sum of the probabilities has to be 1.

On the below, please provide the label you have predicted, which would have the maximum probability value.

### acknowledgement ###
Provide probability value for each of the six classes in the "python dictionary format" on the above.
And the label you have predicted is on the below
Don't Repeat responses you already said. please just let me know the labels and their probability values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)

In [ ]:
### 아래는 잘못 예측한 라벨을 얼만큼 확률을 부여했는지 보는 것.

In [21]:
#test 3 #정답은 improved인데 stable이라고 예측했음(예측 틀린 경우인데, 이 예측 라벨에 대해 얼만큼 확률을 부여했느냐를 확인)
y_i = "stable"
correct = "improved"
report = """
EXAM : Whole spine MRI with contrast enhancement and diffusion weighted image.  
CLINICAL INFORMATION: Breast cancer, Multiple bone metastasis.     
COMPARISON : 2018-03-08 MRI. 
    
FINDINGS :   
No remarkable change of marrow replacing enhancing lesions in  C, T, L, S- spines, sternum, and iliac bones.
- Decreased enhancement of the lesions.  
No evidence of significant cord compression in this study.    
    
CONCLUSION :   
No change of multiple bone metastases.    
- Decreased in enhancement.
   
RECOMMENDATION : Clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Actually, the true label is {correct}.
Based on the predicted label, true label, and the report, provide the condfidence value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the confidence value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the confidence value of the label {y_i} which you have predicted, must be the maximum among all confidence values.
>>> in other words, the other confidence values must be smaller than the confidence of the label you have predicted.



### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide confidence value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their confidence values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


improved: 0.9
mets: 0.8
progression: 0.7
stable: 0.6
no: 0.5
romets: 0.4

### instruction ###
You are a radiologist. you got an important task


In [17]:
#test 4 #정답은 progression인데 mets이라고 예측했음(예측 틀린 경우인데, 이 예측 라벨에 대해 얼만큼 확률을 부여했느냐를 확인)
y_i = "mets"
report = """
EXAM : Whole spine MRI with contrast enhancement 
HISTORY :  Prostate cancer with multiple bone metastases 
 
Compared to previous MRI taken on 2019.08.16, 
 
FINDING :  
Newly developed bone marrow replacing lesions with enhancement in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
Not well delineation of previous noted enhancing lesion in the lamina of C2. 
 
Spondylolisthesis of C4 over C5. 
Retrolisthesis of C6 over C7. 
Mild kyphotic deformity and mild central canal stenosis on C3-4-5-6-7 level. 
 
Disc bulging in T12/L1, L2/3, L3/4, and L4/5. 
- Mild central canal narrowing and bilateral neural foraminal narrowing in L3/4. 
- Bilateral moderate neural foraminal narrowing in L4/5. 
 
Spondylolytic spondylolisthesis of L5 over S1. 
- Bilateral severe neural foraminal narrowing. 
 
IMPRESSION :  
1. Probable bone metastases in T1 vertebral body, left lamina and pedicle of L5, left sacrum and left ilium.
2. Not well delineation of previous noted enhancing lesion in the lamina of C2. 
3. Spondylolytic spondylolisthesis of L5 over S1 with bilateral severe neural foraminal narrowing. 
 
RECOMMEND : clinical correlation.
"""


prompt = f"""
### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label '{y_i}' after you read this radiology report below:

### report ###
{report}

### response format ###.
Please Provide probability value for each of the six labels in the "python dictionary" format.
>>> left of ':' is the label and the right of ':' is the probability value of the label. please answer each label in lowercase. 
>>> especially, do not put the letter R/O to the left of the label "romets".
>>> the probability of the label {y_i} which you have predicted, must be the maximum among all probabilities.
>>> in other words, the other probability values must be smaller than the probability of the label you have predicted
>>> also, sum of the probabilities has to be 1.


### acknowledgement ###
Remember that the label you have predicted is {y_i}. 
Provide probability value for each of the six classes in the "python dictionary format".
Don't Repeat responses you already said. please just let me know the labels and their probability values.

### response ###

"""

response = generate_custom_response(prompt)
print(response)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



### instruction ###
You are a radiologist. you got an important task to predict the label based on the radiology report.
There were only "six" labels you had to predict : "no", "mets", "progression", "stable", "improved", "romets"
You have predicted the label'mets' after you read this radiology report below:

### report ###

EXAM : Whole spine MRI with contrast enhancement 
HISTORY :  Prostate cancer with multiple bone metastases 
 
Compared to previous MRI taken on 2019.08.16, 
 
FINDING :  
Newly developed bone marrow replacing lesions with enhancement in T1 vertebral body, left lamina and pedicle of L5, left sacrum and ilium. 
Not well delineation of previous noted enhancing lesion in the lamina of C2. 
 
Spondylolisthesis of C4 over C5. 
Retrolisthesis of C6 over C7. 
Mild kyphotic deformity and mild central canal stenosis on C3-4-5-6-7 level. 
 
Disc bulging in T12/L1, L2/3, L3/4, and L4/5. 
- Mild central canal narrowing and bilateral neural foraminal narrowing in L3/4. 
- Bilat

In [20]:
########### 추론하기 ############
key = "Based on the report below, classify a cancer treatment response into one of six labels: improved, mets, no, progression, romets, stable.\n\n- Description of six labels\n    - 'no'\n        - a state of being unresponsive to treatment\n        - May mean cancer remains intact or remains unchanged\n    - 'mets'\n        - a state in which a transition has occurred\n        - If the cancer has spread from its original location to another site\n    - 'progression'\n        - a state of progression of cancer\n        - If the cancer has grown or worsened\n    - 'stable'\n        - a stable condition of cancer\n        - If the cancer is no longer progressing and remains in place\n    - 'improved'\n        - a state of improvement in cancer\n        - If the size or condition of the cancer improves\n    - 'romets'\n        - a state of reduced transition\n        - If the original metastatic cancer is reduced\n"
result_save_path = '/root/modelfile/'

generator = pipeline("text-generation", model=new_model, tokenizer=tokenizer) #new_model: 파라미터 합친 모델

def evaluate_model(test_dataset, model, tokenizer):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    def generate_response(batch):
        prompts = [f"### Instruction:\n{key}\n\n### Input:\n{input_text}\n\n### Instruction:\n{key}\n\n### Response:" for input_text in batch['input']]
        responses = generator(prompts, max_new_tokens=5, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        generated_texts = [response[0]['generated_text'].replace(prompt, "").strip() for response, prompt in zip(responses, prompts)]
        return {'predicted': generated_texts}

    test_dataset = test_dataset.map(generate_response, batched=True, batch_size=1)  # Adjust batch_size as needed

    correct = test_dataset['output']
    predicted = test_dataset['predicted']

    # Define the regular expression pattern to match the specified labels
    pattern = re.compile(r'\b(?:no|mets|progression|stable|improved|romets)\b')  # Regular expression pattern

    # Filter the predicted labels using the regex pattern
    filtered = [(c, pattern.search(p).group(0)) for c, p in zip(correct, predicted) if pattern.search(p)]  # Using regex to filter and extract labels
    print('filtered:', filtered)
    
    # Save filtered correct and predicted labels to a CSV file
    filtered_df = pd.DataFrame(filtered, columns=['correct', 'predicted'])  # Creating dataframe with filtered data
    results_csv_path = f'{result_save_path}/result/results_{timestamp}.csv'
    filtered_df.to_csv(results_csv_path, index=False)
    
    # LabelEncoder
    le = LabelEncoder()
    le.fit([c for c, _ in filtered] + [p for _, p in filtered])  # Using filtered values for encoding
    true_labels = le.transform([c for c, _ in filtered])
    predicted_labels = le.transform([p for _, p in filtered])

    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, average='macro')
    recall = recall_score(true_labels, predicted_labels, average='macro')
    f1 = f1_score(true_labels, predicted_labels, average='macro')

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

    # Save metrics to a unique file
    metrics_filename = f'{result_save_path}/result/metrics_{timestamp}.json'
    with open(metrics_filename, 'w') as f:
        json.dump(metrics, f)

    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1 Score: {f1:.4f}')

# Evaluate the model
evaluate_model(test_dataset, model, tokenizer)

'/root/codes'